In [1]:
import kagglehub
import os
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
path = "./data"
print("Path to dataset files : ",path)

Path to dataset files :  ./data


In [ ]:
# Paths to the CSV files
train_csv_path = os.path.join(path, 'train.csv')
test_csv_path = os.path.join(path, 'test.csv')

# Path to the folder containing the actual images
# Based on the image, it looks like 'faces-spring-2020' is the image directory
image_dir = os.path.join(path, 'faces')

# Load the training data to verify the path
df_train = pd.read_csv(train_csv_path)
df_train['id'] = df_train['id'].astype(int).astype(str)
df_train['id'] = 'face-'+df_train['id']+'.png'

df_test = pd.read_csv(test_csv_path)
df_test['id'] = df_test['id'].astype(int).astype(str)
df_test['id'] = 'face-'+df_test['id']+'.png'

print(f"Total images found in train.csv: {len(df_train)}")
print(f"Total images found in train.csv: {len(df_test)}")

Total images found in train.csv: 4500
Total images found in train.csv: 500


In [4]:
final_dim = 64
processed_dir = path + "/faces_processed_" + str(final_dim)
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)


In [5]:
import os
import cv2
import pandas as pd

# Define your interval (e.g., every 50 images)
progress_interval = 50 
count = 0

# Correcting the concat syntax: it needs a list [df_train, df_test]
combined_df = pd.concat([df_train, df_test])
total_images = len(combined_df)

print(f"Resizing images to {final_dim}x{final_dim}...")

for index, row in combined_df.iterrows():
    filename = row['id']
    input_path = os.path.join(image_dir, filename)
    output_path = os.path.join(processed_dir, filename)
    
    # 1. Check if the file already exists in the output directory
    if os.path.exists(output_path):
        count += 1
        continue # Skip to the next image
    
    img = cv2.imread(input_path)
    
    if img is not None:
        resized_img = cv2.resize(img, (final_dim, final_dim), interpolation=cv2.INTER_AREA)
        cv2.imwrite(output_path, resized_img)
        count += 1
        
        # 2. Progress indicator with custom counter
        if count % progress_interval == 0 or count == total_images:
            print(f"Progress: {count}/{total_images} images processed.")
    else:
        print(f"Warning: Could not read {filename} at {input_path}")
        count += 1

print(f"Successfully saved resized images to {processed_dir}")

Resizing images to 64x64...
Successfully saved resized images to ./data/faces_processed_64


In [11]:
import os
import cv2
import pandas as pd
from tqdm.notebook import tqdm

# 1. Define your paths
csv_path = 'data/train.csv' # Or 'data/cleaned_train.csv' if you used the cleaning script
source_dir = 'data/faces' # Change this if your images are in a different folder
dest_dir = 'data/labelled-faces'

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# 2. Load the dataframe and limit it to the first 4500 rows
df = pd.read_csv(csv_path)
df = pd.read_csv(train_csv_path)
df['id'] = df['id'].astype(int).astype(str)
df['id'] = 'face-'+df['id']+'.png'
df_subset=df.head(4500)
print(f"Starting generation of {len(df_subset)} tagged images...")

# 3. Loop through the subset and tag the images
for index, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Tagging Images"):
    filename = row['id']
    label = int(row['glasses'])
    
    # Read the original image
    img_path = os.path.join(source_dir, filename)
    img = cv2.imread(img_path)
    
    if img is None:
        print(f"Warning: Could not read {img_path}. Skipping.")
        continue
        
    # Define the text and positioning
    # Using 'G: 1' or 'G: 0' to save space on small resolution images
    text = f"G: {label}"
    position = (2, 12) # (x, y) coordinates near the top left
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.35 # Small enough to fit on a 64x64 image
    color = (0, 255, 0) # Green text in BGR format
    thickness = 1
    
    # Draw the text on the image
    cv2.putText(img, text, position, font, font_scale, color, thickness)
    
    # Save the tagged image to the new folder
    dest_path = os.path.join(dest_dir, filename)
    cv2.imwrite(dest_path, img)

print(f"Successfully saved tagged images to {dest_dir}")

Starting generation of 4500 tagged images...


Tagging Images:   0%|          | 0/4500 [00:00<?, ?it/s]

Successfully saved tagged images to data/labelled-faces


In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# Ensure this matches your column name (e.g., 'glasses' or 'has_glasses')
label_col = 'glasses' 

def easy_correct(df, batch_size=12):
    current_idx = 0
    print("--- Data Correction Mode ---")
    print("Instructions:")
    print("- Press ENTER to see the next batch.")
    print("- Type an index (e.g., '42') to flip its label.")
    print("- Type 'stop' to finish and save.")

    while current_idx < len(df):
        plt.figure(figsize=(15, 10))
        batch_indices = range(current_idx, min(current_idx + batch_size, len(df)))
        
        for i, idx in enumerate(batch_indices):
            row = df.iloc[idx]
            img_path = os.path.join(processed_dir, row['id'])
            img = cv2.imread(img_path)
            
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.subplot(3, 4, i + 1)
                plt.imshow(img)
                plt.title(f"Idx: {idx} | Label: {row[label_col]}")
                plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        user_cmd = input("Action (Index to flip / Enter for next / 'stop'): ").strip().lower()
        
        if user_cmd == 'stop':
            break
        elif user_cmd.isdigit():
            target_idx = int(user_cmd)
            # Flip the label: 0 becomes 1, 1 becomes 0
            old_val = df.at[target_idx, label_col]
            df.at[target_idx, label_col] = 1 - old_val
            print(f"!!! Corrected Index {target_idx}: {old_val} -> {df.at[target_idx, label_col]}")
            # Do not increment current_idx so you can see the change in the same batch
            continue 
        
        current_idx += batch_size

# Run the tool
easy_correct(combined_df)

In [ ]:
class FacesDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, label_col='glasses'):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # 1. Get the filename and label from your pandas dataframe
        img_name = self.dataframe.iloc[idx]['id'] 
        label = self.dataframe.iloc[idx][self.label_col]

        # 2. Load the image
        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path)
        
        # Convert BGR (OpenCV default) to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 3. Apply transformations (like converting to PyTorch Tensor)
        if self.transform:
            image = self.transform(image)

        # 4. Return the image tensor and the label
        return image, torch.tensor(label, dtype=torch.float32)

In [ ]:
# The ToTensor transform automatically converts the numpy array to a CHW tensor 
# and scales the pixels to a 0-1 range.
import torchvision.transforms as transforms

# Define the aggressive (but safe) augmentation pipeline
augmented_transform = transforms.Compose([
    transforms.ToPILImage(), # Required because OpenCV returns NumPy arrays
    
    # 1. Rotation, Translation, and Scaling
    # Rotates up to 12 degrees, shifts by 5%, scales between 90% and 110%
    transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.9, 1.1)),
    
    # 2. Random Cropping
    # Pads the edges by 4 pixels by reflecting the border, then crops a random 64x64 section
    transforms.RandomCrop(64, padding=4, padding_mode='reflect'),
    
    # 3. Flipping
    # Faces are mostly symmetrical, so this instantly doubles your dataset variations
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 4. Color Jitter
    # Slight changes to lighting and contrast to prevent overfitting to specific photo conditions
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    
    transforms.ToTensor() # Converts back to [0.0, 1.0] tensor
])

# Update your dataset to use this new transform
train_dataset = FacesDataset(
    dataframe=combined_df, 
    image_dir=processed_dir, 
    transform=augmented_transform,
    label_col='glasses' 
)


# Create the DataLoader
batch_size = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, # Shuffle the data every epoch for better generalization
    num_workers=2, # Uses background CPU threads to load images faster
    pin_memory=True # Speeds up transfer from CPU RAM to GPU VRAM
)

print(f"Total batches per epoch: {len(train_loader)}")

In [ ]:
import matplotlib.pyplot as plt

# Fetch one specific image and label from the raw dataset (bypassing the transform temporarily)
sample_img_name = df_train.iloc[0]['id']
import os, cv2
raw_img = cv2.imread(os.path.join(processed_dir, sample_img_name))
raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15, 3))

# Plot Original
plt.subplot(1, 6, 1)
plt.imshow(raw_img)
plt.title("Original")
plt.axis('off')

# Plot 5 Augmented Versions
for i in range(5):
    # Apply the transform
    aug_tensor = augmented_transform(raw_img)
    # Convert back to HWC format for matplotlib
    aug_img = aug_tensor.permute(1, 2, 0).numpy()
    
    plt.subplot(1, 6, i+2)
    plt.imshow(aug_img)
    plt.title(f"Augmented {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from models.vae import VAE
import torch

# 1. Define the device (RTX 3060)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Initialize the model with your desired hyperparameters
# This is where you can change latent_dim or kernel_size for your ablation table
model = VAE(latent_dim=128, kernel_size=3).to(device)

# 3. Now run your dummy test
dummy_input = torch.randn(4, 3, 64, 64).to(device)
recon_batch, mu, logvar = model(dummy_input)

print(f"Success! Output shape: {recon_batch.shape}")

In [ ]:
import torch
from torch import optim
from tqdm.notebook import tqdm # Use notebook version for cleaner UI
import matplotlib.pyplot as plt

# 1. Setup Optimizer and Epochs
# A learning rate of 1e-3 is standard for Adam with VAEs
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 50 # Start with 15 to see how long it takes on your GPU

# List to store average loss per epoch for your ablation plots
epoch_losses = []

print(f"Starting training on {device}...")

for epoch in range(epochs):
    model.train()
    train_loss = 0
    
    # Wrap your loader in tqdm for the progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    
    for batch_idx, (data, labels) in enumerate(progress_bar):
        # Move the image batch to the RTX 3060 VRAM
        data = data.to(device)
        
        # Inject Gaussian Noise (mean 0, std 0.1)
        # torch.clamp ensures pixel values don't exceed the [0, 1] bounds
        noisy_data = data + torch.randn_like(data) * 0.1
        noisy_data = torch.clamp(noisy_data, 0., 1.)
        # Reset the gradients
        optimizer.zero_grad()
        
        # Forward pass
        recon_batch, mu, logvar = model(data)
        
        # Calculate loss (Ensure vae_loss_function is imported)
        loss = vae_loss_function(recon_batch, data, mu, logvar)
        
        # Backward pass (calculate gradients)
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        train_loss += loss.item()
        
        # Update the progress bar with the current batch loss
        # We divide by batch size (data.size(0)) to get average loss per image
        progress_bar.set_postfix({'Loss': f"{loss.item() / data.size(0):.4f}"})
        
    # Calculate average loss for the entire epoch
    avg_loss = train_loss / len(train_loader.dataset)
    epoch_losses.append(avg_loss)
    
    print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f}")

print("Training finished!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a professional seaborn theme
sns.set_theme(style="whitegrid")

# Create the figure
plt.figure(figsize=(10, 6))

# Plot the epoch_losses list we populated during training
sns.lineplot(
    x=range(1, len(epoch_losses) + 1), 
    y=epoch_losses, 
    marker='o', 
    linewidth=2.5, 
    markersize=8,
    color='#2b5b84' # A clean, academic blue
)

# Format the labels and title
plt.title('VAE Training Loss Convergence', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Epoch', fontsize=12, fontweight='bold')
plt.ylabel('Average Loss (BCE + KLD)', fontsize=12, fontweight='bold')

# Ensure the x-axis only shows whole integer epochs
plt.xticks(range(1, len(epoch_losses) + 1))

plt.tight_layout()

# Save the plot to your directory so you can drop it into your report
plt.savefig('vae_training_loss.png', dpi=300)

# Display the plot in the notebook
plt.show()

In [ ]:
# Switch model to evaluation mode (turns off dropout/batchnorm if you add them later)
model.eval()

with torch.no_grad(): # Do not track gradients for generation
    # Sample 6 random vectors from a standard normal distribution
    # latent_dim should match what you initialized the model with (e.g., 128)
    sample_z = torch.randn(6, 128).to(device)
    
    # Pass them through the decoder
    generated_images = model.decoder(model.fc_decode(sample_z)).cpu()

    # Plot the results
    plt.figure(figsize=(15, 5))
    for i in range(6):
        plt.subplot(1, 6, i+1)
        # Permute from PyTorch format (C, H, W) to Matplotlib format (H, W, C)
        img_to_show = generated_images[i].permute(1, 2, 0).numpy()
        plt.imshow(img_to_show)
        plt.axis('off')
        plt.title(f"Generated {i+1}")
        
    plt.tight_layout()
    plt.show()

In [ ]:
def get_average_latents(model, dataloader, device):
    model.eval()
    mu_glasses = []
    mu_no_glasses = []
    
    print("Extracting latent vectors...")
    with torch.no_grad():
        for data, labels in dataloader:
            data = data.to(device)
            # Pass through the encoder and the mu linear layer
            encoded = model.encoder(data)
            mu = model.fc_mu(encoded)
            
            # Sort by label
            for i in range(len(labels)):
                if labels[i].item() == 1.0: # 1 = Glasses
                    mu_glasses.append(mu[i])
                else:
                    mu_no_glasses.append(mu[i])
                    
    # Stack the lists into tensors and calculate the mean across all images
    avg_glasses = torch.stack(mu_glasses).mean(dim=0)
    avg_no_glasses = torch.stack(mu_no_glasses).mean(dim=0)
    
    return avg_glasses, avg_no_glasses

# 1. Get the average vectors
avg_glasses, avg_no_glasses = get_average_latents(model, train_loader, device)

# 2. Create 3 variations for each class
# We duplicate the average vector 3 times, then add a small amount of random noise (scale=0.5) 
# to generate distinct faces that share the core "glasses" or "no glasses" traits.
latent_dim = 128 # Ensure this matches your model
noise_scale = 0.5

z_glasses = avg_glasses.unsqueeze(0).repeat(3, 1) + (torch.randn(3, latent_dim).to(device) * noise_scale)
z_no_glasses = avg_no_glasses.unsqueeze(0).repeat(3, 1) + (torch.randn(3, latent_dim).to(device) * noise_scale)

# Combine them into one batch of 6
z_combined = torch.cat([z_glasses, z_no_glasses], dim=0)

# 3. Decode and visualize
with torch.no_grad():
    generated_images = model.decoder(model.fc_decode(z_combined)).cpu()

plt.figure(figsize=(15, 6))
for i in range(6):
    plt.subplot(2, 3, i+1)
    img_to_show = generated_images[i].permute(1, 2, 0).numpy()
    plt.imshow(img_to_show)
    plt.axis('off')
    title = "Glasses" if i < 3 else "No Glasses"
    plt.title(f"{title} - Person {i%3 + 1}")

plt.suptitle("VAE Generation: Glasses vs No Glasses", fontsize=16)
plt.tight_layout()
plt.show()